In [8]:
import sys
from pathlib import Path
import requests
from bs4 import BeautifulSoup
import pandas as pd
from requests import HTTPError

# Add project root to path (if needed for shared utils later)
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

In [9]:
def generate_211_url(latitude, longitude, location, topic_path):
    base_url = "https://211ontario.ca/results/"
    return (
        f"{base_url}?latitude={latitude}&longitude={longitude}&searchLocation={location}"
        f"&searchTerms=&exct=0&sd=25&ss=Distance&topicPath={topic_path}"
    )

In [10]:
ontario_city_coordinates = {
    "Barrie": {"lat": 44.3894, "lon": -79.6903},
    "Belleville": {"lat": 44.1628, "lon": -77.3832},
    "Brampton": {"lat": 43.7315, "lon": -79.7624},
    "Brant": {"lat": 43.1517, "lon": -80.4287},
    "Brantford": {"lat": 43.1414, "lon": -80.2700},
    "Brockville": {"lat": 44.5902, "lon": -75.6830},
    "Burlington": {"lat": 43.3255, "lon": -79.7990},
    "Cambridge": {"lat": 43.3616, "lon": -80.3144},
    "Clarence-Rockland": {"lat": 45.5510, "lon": -75.2894},
    "Cornwall": {"lat": 45.0213, "lon": -74.7303},
    "Dryden": {"lat": 49.7801, "lon": -92.8370},
    "Elliot Lake": {"lat": 46.3865, "lon": -82.6510},
    "Greater Sudbury": {"lat": 46.4917, "lon": -81.0140},
    "Guelph": {"lat": 43.5448, "lon": -80.2482},
    "Haldimand County": {"lat": 42.9566, "lon": -79.8732},
    "Hamilton": {"lat": 43.2557, "lon": -79.8711},
    "Kawartha Lakes": {"lat": 44.3600, "lon": -78.7400},
    "Kenora": {"lat": 49.7667, "lon": -94.4837},
    "Kingston": {"lat": 44.2312, "lon": -76.4860},
    "Kitchener": {"lat": 43.4504, "lon": -80.4832},
    "London": {"lat": 42.9849, "lon": -81.2453},
    "Markham": {"lat": 43.8561, "lon": -79.3370},
    "Mississauga": {"lat": 43.5890, "lon": -79.6441},
    "Niagara Falls": {"lat": 43.0896, "lon": -79.0849},
    "Norfolk County": {"lat": 42.8023, "lon": -80.4074},
    "North Bay": {"lat": 46.3091, "lon": -79.4608},
    "Orillia": {"lat": 44.6082, "lon": -79.4186},
    "Oshawa": {"lat": 43.8971, "lon": -78.8658},
    "Ottawa": {"lat": 45.4215, "lon": -75.6972},
    "Owen Sound": {"lat": 44.5690, "lon": -80.9406},
    "Pembroke": {"lat": 45.8267, "lon": -77.1109},
    "Peterborough": {"lat": 44.3091, "lon": -78.3197},
    "Pickering": {"lat": 43.8384, "lon": -79.0868},
    "Port Colborne": {"lat": 42.8875, "lon": -79.2497},
    "Prince Edward County": {"lat": 44.0001, "lon": -77.2500},
    "Quinte West": {"lat": 44.1026, "lon": -77.5818},
    "Richmond Hill": {"lat": 43.8828, "lon": -79.4403},
    "Sarnia": {"lat": 42.9745, "lon": -82.4066},
    "Sault Ste. Marie": {"lat": 46.5136, "lon": -84.3358},
    "St. Catharines": {"lat": 43.1594, "lon": -79.2469},
    "St. Thomas": {"lat": 42.7789, "lon": -81.1928},
    "Stratford": {"lat": 43.3708, "lon": -80.9822},
    "Temiskaming Shores": {"lat": 47.5168, "lon": -79.6780},
    "Thorold": {"lat": 43.1169, "lon": -79.2400},
    "Thunder Bay": {"lat": 48.3809, "lon": -89.2477},
    "Timmins": {"lat": 48.4758, "lon": -81.3289},
    "Toronto": {"lat": 43.6532, "lon": -79.3832},
    "Vaughan": {"lat": 43.8563, "lon": -79.5085},
    "Waterloo": {"lat": 43.4643, "lon": -80.5204},
    "Welland": {"lat": 42.9930, "lon": -79.2483},
    "Windsor": {"lat": 42.3149, "lon": -83.0364},
    "Woodstock": {"lat": 43.1315, "lon": -80.7472}
}

In [11]:
# Browser automation with Playwright to pass Turnstile and paginate
# If not installed, run in a separate cell/terminal: pip install playwright && playwright install
import asyncio
from urllib.parse import urljoin
from playwright.async_api import async_playwright

def parse_records(html, source_url):
    soup = BeautifulSoup(html, "html.parser")
    resources = []
    for record in soup.select("div.record"):
        title_el = record.select_one(".title a")
        title = title_el.get_text(strip=True) if title_el else None
        subtitle_el = record.select_one(".subtitle")
        subtitle = subtitle_el.get_text(strip=True) if subtitle_el else None
        city_el = record.select_one(".subsubtitle")
        city = city_el.get_text(strip=True) if city_el else None
        phone_numbers = [a.get_text(strip=True) for a in record.select("span.linkphone_fa6 a")]
        website_el = record.select_one("a.linkexternal_fa6")
        website = website_el["href"] if website_el and website_el.has_attr("href") else None
        record_info = record.find_next_sibling("div", class_="record-info")
        address = None
        if record_info:
            info_text = " ".join(record_info.stripped_strings)
            address = info_text.replace("(0km)", "").strip() if info_text else None
        details = record.find_next_sibling("div", class_="record-list-details")
        description = None
        if details:
            desc_el = details.select_one(".record-list-desc")
            description = desc_el.get_text(" ", strip=True) if desc_el else None
        resources.append({
            "name": title,
            "program": subtitle,
            "address": address,
            "city": city,
            "phone": ", ".join(phone_numbers) if phone_numbers else None,
            "website": website,
            "description": description,
            "source_url": source_url
        })
    return resources

async def fetch_all_pages(start_url):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context()
        page = await context.new_page()
        await page.goto(start_url, wait_until="networkidle")
        print("If a Turnstile challenge appears, complete it in the browser window.")
        await page.wait_for_timeout(10000)
        html = await page.content()
        soup = BeautifulSoup(html, "html.parser")
        pagination_links = []
        for a in soup.select("div.paging a[href]"):
            href = a.get("href")
            if not href:
                continue
            pagination_links.append(urljoin(start_url, href))
        # Ensure first page included and dedupe while keeping order
        all_pages = [start_url] + pagination_links
        seen = set()
        ordered_pages = []
        for u in all_pages:
            if u not in seen:
                seen.add(u)
                ordered_pages.append(u)
        results = []
        for u in ordered_pages:
            await page.goto(u, wait_until="networkidle")
            await page.wait_for_timeout(3000)
            results.append((u, await page.content()))
        await browser.close()
        return results

# Loop through all Ontario cities and collect results
topic_path = 123  # Replace with mental health/addiction subtype from your lookup table
pause_seconds = 8
all_records = []

for location, coords in ontario_city_coordinates.items():
    latitude = coords["lat"]
    longitude = coords["lon"]
    url = generate_211_url(latitude, longitude, location, topic_path)
    pages = await fetch_all_pages(url)
    print(f"{location}: Pages fetched: {len(pages)}")
    for page_url, html in pages:
        # Save each page HTML if you want to inspect
        output_path = project_root / "test" / "extraction_test" / "ontario_211_results.html"
        output_path.write_text(html, encoding="utf-8")
        all_records.extend(parse_records(html, page_url))
    # Stop gap to avoid flooding servers
    await asyncio.sleep(pause_seconds)

df = pd.DataFrame(all_records)
csv_path = project_root / "test" / "extraction_test" / "ontario_211_mental_health_addiction.csv"
df.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path)

If a Turnstile challenge appears, complete it in the browser window.
Barrie: Pages fetched: 2
If a Turnstile challenge appears, complete it in the browser window.
Belleville: Pages fetched: 2
If a Turnstile challenge appears, complete it in the browser window.
Brampton: Pages fetched: 3
If a Turnstile challenge appears, complete it in the browser window.
Brant: Pages fetched: 2
If a Turnstile challenge appears, complete it in the browser window.
Brantford: Pages fetched: 2
If a Turnstile challenge appears, complete it in the browser window.
Brockville: Pages fetched: 2
If a Turnstile challenge appears, complete it in the browser window.
Burlington: Pages fetched: 3
If a Turnstile challenge appears, complete it in the browser window.
Cambridge: Pages fetched: 3
If a Turnstile challenge appears, complete it in the browser window.
Clarence-Rockland: Pages fetched: 2
If a Turnstile challenge appears, complete it in the browser window.
Cornwall: Pages fetched: 2
If a Turnstile challenge app